In [ ]:
# ============================================================
# INSTALLS (Colab-safe)
# ============================================================
!pip install -q pandas requests openpyxl tqdm pycountry pycountry-convert

# ============================================================
# IMPORTS & CONFIG
# ============================================================
import pandas as pd
import requests
import time
import re
from tqdm import tqdm
from google.colab import files
import getpass
import pycountry
import pycountry_convert as pc

TMDB_KEY  = getpass.getpass("TMDb API key (v3): ")
TMDB_BASE = "https://api.themoviedb.org/3"
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

SLEEP_TMDB = 0.25
SLEEP_WD   = 0.35  # um pouco mais alto para não irritar a Wikidata

# ============================================================
# LOAD FILE (dataset já enriquecido pela versão anterior)
# ============================================================
uploaded = files.upload()
FNAME = list(uploaded.keys())[0]
df = pd.read_excel(FNAME)
print("Loaded:", FNAME, "Rows:", len(df))

# ============================================================
# COLUMN NAMES (já existem)
# ============================================================
COL_TITLE     = "Movie Title"
COL_DATE      = "Date"

COL_CITY      = "Production Company City"
COL_CONTINENT = "Production Company Continent"

# isto pode existir (não mexemos), mas pode ajudar em fallback
COL_COUNTRY_EXISTING = "Production Company Country"
COL_COMPANY_NAME     = "Production Studio"  # ajusta aqui se a tua coluna tiver outro nome

# Confirmações mínimas
missing_cols = [c for c in [COL_TITLE, COL_DATE, COL_CITY, COL_CONTINENT] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Estas colunas não existem no teu ficheiro: {missing_cols}")

# ============================================================
# HELPERS HTTP (com retries + backoff)
# ============================================================
def http_get(url, params=None, headers=None, timeout=30, max_tries=4, base_sleep=1.0):
    back = base_sleep
    for _ in range(max_tries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(back)
                back *= 1.8
                continue
            return r
        except requests.RequestException:
            time.sleep(back)
            back *= 1.8
    return None

def http_post(url, data=None, headers=None, timeout=45, max_tries=4, base_sleep=1.0):
    back = base_sleep
    for _ in range(max_tries):
        try:
            r = requests.post(url, data=data, headers=headers, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(back)
                back *= 1.8
                continue
            return r
        except requests.RequestException:
            time.sleep(back)
            back *= 1.8
    return None

def get_year(date_val):
    try:
        y = pd.to_datetime(date_val, errors="coerce").year
        if pd.isna(y): return None
        return int(y)
    except:
        return None

def normalize_title(t):
    t = str(t).lower().strip()
    t = re.sub(r"\s*\(\d{4}\)", "", t)
    t = re.sub(r"[^a-z0-9 ]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def parse_city_from_headquarters(hq):
    if not isinstance(hq, str):
        return None
    hq = hq.strip()
    if not hq:
        return None
    # "Burbank, California" -> "Burbank"
    return hq.split(",")[0].strip() if "," in hq else hq

# ============================================================
# ISO2 -> CONTINENT (global, robust)
# ============================================================
CONTINENT_CODE_TO_NAME = {
    "AF": "Africa",
    "NA": "North America",
    "SA": "South America",
    "AS": "Asia",
    "EU": "Europe",
    "OC": "Oceania",
    "AN": "Antarctica",
}

def iso2_to_continent_name(iso2):
    if not isinstance(iso2, str) or len(iso2) != 2:
        return None
    iso2 = iso2.upper()
    try:
        cont_code = pc.country_alpha2_to_continent_code(iso2)
        return CONTINENT_CODE_TO_NAME.get(cont_code)
    except Exception:
        return None

def country_name_to_iso2(country_name):
    if not isinstance(country_name, str) or not country_name.strip():
        return None
    try:
        obj = pycountry.countries.lookup(country_name.strip())
        return obj.alpha_2
    except Exception:
        return None

# ============================================================
# TMDb caches
# ============================================================
movie_id_cache = {}            # (title_norm, year) -> movie_id
movie_companies_cache = {}     # movie_id -> list of companies [{id,name},...]
company_details_cache = {}     # company_id -> {"city":..., "iso2":...}

def tmdb_search_movie_id(title, year=None):
    key = (normalize_title(title), year)
    if key in movie_id_cache:
        return movie_id_cache[key]

    params = {"api_key": TMDB_KEY, "query": title, "include_adult": "false"}
    if year:
        params["year"] = int(year)

    r = http_get(f"{TMDB_BASE}/search/movie", params=params, timeout=25)
    if not r or r.status_code >= 400:
        movie_id_cache[key] = None
        return None

    res = (r.json() or {}).get("results") or []
    if not res:
        movie_id_cache[key] = None
        return None

    mid = res[0].get("id")
    movie_id_cache[key] = mid
    time.sleep(SLEEP_TMDB)
    return mid

def tmdb_get_movie_companies(movie_id):
    if movie_id in movie_companies_cache:
        return movie_companies_cache[movie_id]

    r = http_get(f"{TMDB_BASE}/movie/{movie_id}", params={"api_key": TMDB_KEY}, timeout=25)
    if not r or r.status_code >= 400:
        movie_companies_cache[movie_id] = []
        return []

    pcs = (r.json() or {}).get("production_companies") or []
    out = [{"id": c.get("id"), "name": c.get("name")} for c in pcs if c.get("id") is not None]
    movie_companies_cache[movie_id] = out
    time.sleep(SLEEP_TMDB)
    return out

def tmdb_get_company_details(company_id):
    if company_id in company_details_cache:
        return company_details_cache[company_id]

    r = http_get(f"{TMDB_BASE}/company/{company_id}", params={"api_key": TMDB_KEY}, timeout=25)
    city = None
    iso2 = None
    if r and r.status_code < 400:
        j = r.json() or {}
        city = parse_city_from_headquarters(j.get("headquarters"))
        iso2 = j.get("origin_country")  # ISO2
    company_details_cache[company_id] = {"city": city, "iso2": iso2}
    time.sleep(SLEEP_TMDB)
    return company_details_cache[company_id]

# ============================================================
# Wikidata (fallback) caches
# ============================================================
wd_company_cache = {}  # company_name_lower -> {"city":..., "iso2":..., "continent":...}

def wikidata_company_hq(company_name):
    """
    Tries to resolve:
    - HQ city label (best effort)
    - country ISO2 (best effort)
    - continent name (derived from ISO2, if available)
    Uses SPARQL with a pragmatic entity resolution strategy:
      1) Match by label (English) for organizations
      2) Pull headquarters (P159) -> city label
      3) Pull country (P17) -> ISO2 if possible
    """
    if not isinstance(company_name, str) or not company_name.strip():
        return {"city": None, "iso2": None, "continent": None}

    key = company_name.strip().lower()
    if key in wd_company_cache:
        return wd_company_cache[key]

    # SPARQL note:
    # - We try to find an organization with that label.
    # - Then pull headquarters and country.
    query = f"""
    SELECT ?org ?hqLabel ?countryLabel ?iso2 WHERE {{
      ?org rdfs:label "{company_name.strip()}"@en .
      ?org wdt:P31/wdt:P279* wd:Q43229 .   # organization or subclass
      OPTIONAL {{
        ?org wdt:P159 ?hq .
        ?hq rdfs:label ?hqLabel .
        FILTER (lang(?hqLabel) = "en")
      }}
      OPTIONAL {{
        ?org wdt:P17 ?country .
        ?country rdfs:label ?countryLabel .
        FILTER (lang(?countryLabel) = "en")
        OPTIONAL {{ ?country wdt:P297 ?iso2 . }}
      }}
    }}
    LIMIT 5
    """

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "BI-Student-Project/1.0 (contact: academic use)"
    }
    r = http_post(WIKIDATA_SPARQL, data={"query": query}, headers=headers, timeout=60)
    city = None
    iso2 = None
    continent = None

    if r and r.status_code < 400:
        data = r.json()
        bindings = (data.get("results") or {}).get("bindings") or []
        # pick first binding that has HQ or country
        best = None
        for b in bindings:
            if "hqLabel" in b or "iso2" in b or "countryLabel" in b:
                best = b
                break
        if best:
            if "hqLabel" in best:
                city = best["hqLabel"]["value"].strip()
            if "iso2" in best:
                iso2 = best["iso2"]["value"].strip().upper()
            else:
                # fallback: derive iso2 from country label if present
                if "countryLabel" in best:
                    iso2 = country_name_to_iso2(best["countryLabel"]["value"])

    if iso2:
        continent = iso2_to_continent_name(iso2)

    time.sleep(SLEEP_WD)

    wd_company_cache[key] = {"city": city, "iso2": iso2, "continent": continent}
    return wd_company_cache[key]

# ============================================================
# ENRICHMENT: only fill missing City/Continent
# Strategy:
# 1) For each (movie title + year) -> pick "best company" from TMDb:
#    - prefer first company that has iso2 or HQ city
# 2) If City missing -> try Wikidata using company name (from TMDb if available, else your dataset col)
# 3) Continent:
#    - prefer TMDb iso2 -> continent
#    - else Wikidata iso2 -> continent
#    - else fallback from your existing country name column -> iso2 -> continent
# ============================================================
df["_year_tmp"] = df[COL_DATE].apply(get_year)
movie_keys = df[[COL_TITLE, "_year_tmp"]].dropna(subset=[COL_TITLE]).drop_duplicates()

print("Enriching (fill only missing) City + Continent with TMDb + Wikidata...")

# Cache per movie key so we don't repeat work for every daily row
movie_geo_cache = {}  # (title_lower, year) -> {"city":..., "continent":...}

for _, r in tqdm(movie_keys.iterrows(), total=len(movie_keys)):
    title = r[COL_TITLE]
    year  = r["_year_tmp"]
    if not isinstance(title, str) or not title.strip():
        continue

    mkey = (title.strip().lower(), year)
    if mkey in movie_geo_cache:
        continue

    # If there are already many rows filled for this movie, we still compute,
    # but we will only apply where missing later.
    mid = tmdb_search_movie_id(title, year)
    tmdb_city = None
    tmdb_continent = None
    company_name_for_wd = None

    if mid:
        companies = tmdb_get_movie_companies(mid)
        chosen = None
        chosen_det = None

        # prefer a company that gives iso2 OR city
        for c in companies:
            det = tmdb_get_company_details(c["id"])
            if det.get("iso2") or det.get("city"):
                chosen = c
                chosen_det = det
                break

        if chosen is None and companies:
            chosen = companies[0]
            chosen_det = tmdb_get_company_details(chosen["id"])

        if chosen:
            company_name_for_wd = chosen.get("name")

        if chosen_det:
            tmdb_city = chosen_det.get("city")
            iso2 = chosen_det.get("iso2")
            tmdb_continent = iso2_to_continent_name(iso2) if iso2 else None

    # Wikidata fallback only if still missing something important
    wd_city = None
    wd_continent = None

    if (tmdb_city is None or tmdb_continent is None):
        # pick a company name:
        # 1) from TMDb chosen company name
        # 2) else from dataset production studio column if present
        candidate_name = company_name_for_wd
        if (not candidate_name) and (COL_COMPANY_NAME in df.columns):
            # get the most common production studio value for that movie
            # (simple heuristic; avoids random row)
            subset = df.loc[(df[COL_TITLE].astype(str).str.lower().str.strip() == mkey[0]) & (df["_year_tmp"] == year)]
            if len(subset) > 0:
                vc = subset[COL_COMPANY_NAME].dropna().astype(str).value_counts()
                if len(vc) > 0:
                    candidate_name = vc.index[0]

        wd = wikidata_company_hq(candidate_name) if candidate_name else {"city": None, "continent": None}
        wd_city = wd.get("city")
        wd_continent = wd.get("continent")

    final_city = tmdb_city if tmdb_city else wd_city
    final_cont = tmdb_continent if tmdb_continent else wd_continent

    movie_geo_cache[mkey] = {"city": final_city, "continent": final_cont}

# Apply to df (only missing)
for i, row in df.iterrows():
    title = row.get(COL_TITLE)
    year  = row.get("_year_tmp")

    if not isinstance(title, str) or not title.strip():
        continue

    mkey = (title.strip().lower(), year)
    geo = movie_geo_cache.get(mkey)
    if not geo:
        continue

    # Fill City only if missing
    if pd.isna(df.at[i, COL_CITY]) and geo.get("city"):
        df.at[i, COL_CITY] = geo["city"]

    # Fill Continent only if missing
    if pd.isna(df.at[i, COL_CONTINENT]) and geo.get("continent"):
        df.at[i, COL_CONTINENT] = geo["continent"]

    # Fallback from existing country name (your dataset column), if still missing continent
    if pd.isna(df.at[i, COL_CONTINENT]) and (COL_COUNTRY_EXISTING in df.columns):
        iso2_fb = country_name_to_iso2(row.get(COL_COUNTRY_EXISTING))
        cont_fb = iso2_to_continent_name(iso2_fb) if iso2_fb else None
        if cont_fb:
            df.at[i, COL_CONTINENT] = cont_fb

# Cleanup + save
df.drop(columns=["_year_tmp"], inplace=True)

OUT = f"enriched_company_city_continent_WIKIDATA_{FNAME}"
df.to_excel(OUT, index=False)
files.download(OUT)

print("✅ Completed:", OUT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.5/253.5 kB 19.5 MB/s eta 0:00:00
TMDb API key (v3): ··········


Saving enriched_company_city_continent_enriched_COMPANY_CITY_CONTINENT_Final Data Set (1).xlsx to enriched_company_city_continent_enriched_COMPANY_CITY_CONTINENT_Final Data Set (1) (1).xlsx
Loaded: enriched_company_city_continent_enriched_COMPANY_CITY_CONTINENT_Final Data Set (1) (1).xlsx Rows: 34567
Enriching (fill only missing) City + Continent with TMDb + Wikidata...


100%|██████████| 848/848 [16:36<00:00,  1.18s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Completed: enriched_company_city_continent_WIKIDATA_enriched_company_city_continent_enriched_COMPANY_CITY_CONTINENT_Final Data Set (1) (1).xlsx
